# Binary Fall Detection — PoseC3D fine-tuning

This notebook consumes the validated `fall_pose_clean_metadata.csv`, creates leakage-safe grouped splits, converts COCO-17 CSV sequences to MMAction2 annotations, fine-tunes NTU60-pretrained PoseC3D, and evaluates the held-out test set.

Labels: `0 = No Fall`, `1 = Fall`. Enable a Kaggle GPU and internet before starting.

## Before running

Attach both Kaggle inputs: the original `payutch/fall-video-dataset` and a small private Kaggle dataset containing the downloaded `fall_pose_clean_metadata.csv`.

In [ ]:
from pathlib import Path
import json
import pickle
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    average_precision_score, recall_score,
)
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
np.random.seed(SEED)
print('Imports complete.')

## 1. Locate inputs and rebuild Kaggle paths

In [ ]:
DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/payutch/fall-video-dataset'),
    Path('/kaggle/input/fall-video-dataset'),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError('Attach payutch/fall-video-dataset to this notebook.')

metadata_candidates = sorted(Path('/kaggle/input').rglob('fall_pose_clean_metadata.csv'))
working_metadata = Path('/kaggle/working/fall_pose_clean_metadata.csv')
if metadata_candidates:
    CLEAN_METADATA_PATH = metadata_candidates[0]
elif working_metadata.exists():
    CLEAN_METADATA_PATH = working_metadata
else:
    raise FileNotFoundError('Attach a Kaggle input containing fall_pose_clean_metadata.csv.')

ARTIFACT_ROOT = Path('/kaggle/working/posec3d_fall')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
ANNOTATION_PATH = ARTIFACT_ROOT / 'fall_posec3d.pkl'
SPLIT_METADATA_PATH = ARTIFACT_ROOT / 'fall_pose_split_metadata.csv'

print('Dataset:', DATASET_ROOT)
print('Metadata:', CLEAN_METADATA_PATH)
print('Artifacts:', ARTIFACT_ROOT)

In [ ]:
metadata = pd.read_csv(CLEAN_METADATA_PATH)
required = {
    'label', 'normalized_stem', 'csv_relative_path', 'video_relative_path',
    'csv_hash', 'group_id', 'num_frames',
}
missing = required - set(metadata.columns)
if missing:
    raise ValueError(f'Metadata is missing columns: {sorted(missing)}')

metadata['csv_path'] = metadata.csv_relative_path.map(lambda value: str(DATASET_ROOT / value))
metadata['video_path'] = metadata.video_relative_path.map(lambda value: str(DATASET_ROOT / value))
metadata['label'] = metadata.label.astype(int)
metadata['num_frames'] = metadata.num_frames.astype(int)

assert len(metadata) == 6766, f'Expected 6766 clean samples, found {len(metadata)}'
assert metadata.csv_hash.is_unique
assert metadata.num_frames.gt(0).all()
assert metadata.csv_path.map(lambda p: Path(p).exists()).all()
assert metadata.video_path.map(lambda p: Path(p).exists()).all()

display(metadata.head())
display(metadata.label.value_counts().rename_axis('label').reset_index(name='samples'))

## 2. Create optimized grouped 70/15/15 splits

Candidate group splits are searched deterministically. The objective jointly minimizes size error and fall-prevalence error. No `group_id` or exact pose hash may cross splits.

In [ ]:
def best_group_holdout(df, holdout_fraction, attempts=300, seed=42):
    target_positive_rate = df.label.mean()
    best = None
    for attempt in range(attempts):
        splitter = GroupShuffleSplit(n_splits=1, test_size=holdout_fraction, random_state=seed + attempt)
        keep_idx, holdout_idx = next(splitter.split(df, y=df.label, groups=df.group_id))
        holdout = df.iloc[holdout_idx]
        size_error = abs(len(holdout) / len(df) - holdout_fraction)
        prevalence_error = abs(holdout.label.mean() - target_positive_rate)
        score = size_error + prevalence_error
        if best is None or score < best[0]:
            best = (score, keep_idx, holdout_idx)
    return best[1], best[2]

trainval_idx, test_idx = best_group_holdout(metadata, 0.15, seed=SEED)
trainval = metadata.iloc[trainval_idx].copy()
test = metadata.iloc[test_idx].copy()
relative_val_fraction = 0.15 / 0.85
train_idx, val_idx = best_group_holdout(trainval, relative_val_fraction, seed=SEED + 1000)
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train['split'] = 'train'
val['split'] = 'val'
test['split'] = 'test'
split_metadata = pd.concat([train, val, test], ignore_index=True)
split_metadata.to_csv(SPLIT_METADATA_PATH, index=False)

def split_summary(df):
    return df.groupby('split').agg(samples=('label', 'size'), groups=('group_id', 'nunique'), falls=('label', 'sum'), fall_rate=('label', 'mean')).reset_index()
display(split_summary(split_metadata))

In [ ]:
split_names = ['train', 'val', 'test']
for left_index, left in enumerate(split_names):
    for right in split_names[left_index + 1:]:
        left_rows = split_metadata[split_metadata.split == left]
        right_rows = split_metadata[split_metadata.split == right]
        assert set(left_rows.group_id).isdisjoint(set(right_rows.group_id)), f'Group leakage: {left}/{right}'
        assert set(left_rows.csv_hash).isdisjoint(set(right_rows.csv_hash)), f'Hash leakage: {left}/{right}'
for split_name in split_names:
    assert set(split_metadata.loc[split_metadata.split == split_name, 'label']) == {0, 1}
assert len(split_metadata) == len(metadata)
print('Leakage checks passed: groups and pose hashes are disjoint across all splits.')

## 3. Convert COCO-17 CSVs to MMAction2 PoseDataset annotations

In [ ]:
CANONICAL_COCO17 = [
    'Nose', 'Left Eye', 'Right Eye', 'Left Ear', 'Right Ear',
    'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow',
    'Left Wrist', 'Right Wrist', 'Left Hip', 'Right Hip',
    'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle',
]

def reconstruct_csv(csv_path):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={column: column.strip().lower() for column in df.columns})
    required_columns = {'frame', 'keypoint', 'x', 'y', 'confidence'}
    if not required_columns.issubset(df.columns):
        raise ValueError(f'{csv_path} lacks {sorted(required_columns - set(df.columns))}')
    if df.duplicated(['frame', 'keypoint']).any():
        raise ValueError(f'Duplicate frame/keypoint rows: {csv_path}')
    frame_ids = np.sort(df.frame.unique())
    full_index = pd.MultiIndex.from_product([frame_ids, CANONICAL_COCO17], names=['frame', 'keypoint'])
    indexed = df.set_index(['frame', 'keypoint']).reindex(full_index)
    xy = indexed[['x', 'y']].to_numpy(np.float32).reshape(len(frame_ids), 17, 2)
    score = indexed.confidence.fillna(0).to_numpy(np.float32).reshape(len(frame_ids), 17)
    if np.isnan(xy).any():
        raise ValueError(f'Missing pose coordinates in validated CSV: {csv_path}')
    return xy, score

def video_shape(video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not open {video_path}')
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()
    return height, width

def make_annotation(row):
    xy, score = reconstruct_csv(row.csv_path)
    height, width = video_shape(row.video_path)
    sample_id = f'{int(row.label)}_{row.normalized_stem}_{row.csv_hash[:12]}'
    return {
        'frame_dir': sample_id, 'label': int(row.label),
        'img_shape': (height, width), 'original_shape': (height, width),
        'total_frames': len(xy),
        'keypoint': xy[None, ...].astype(np.float16),
        'keypoint_score': score[None, ...].astype(np.float16),
    }

annotations = []
split_ids = {name: [] for name in split_names}
for index, row in enumerate(split_metadata.itertuples(index=False), start=1):
    annotation = make_annotation(row)
    annotations.append(annotation)
    split_ids[row.split].append(annotation['frame_dir'])
    if index % 250 == 0 or index == len(split_metadata):
        print(f'Converted {index}/{len(split_metadata)}', end='\r')

annotation_bundle = {'split': split_ids, 'annotations': annotations}
with open(ANNOTATION_PATH, 'wb') as file:
    pickle.dump(annotation_bundle, file, protocol=pickle.HIGHEST_PROTOCOL)
print(f'\nSaved {len(annotations)} annotations to {ANNOTATION_PATH}')
print('Annotation size (MB):', round(ANNOTATION_PATH.stat().st_size / 1024**2, 2))

In [ ]:
with open(ANNOTATION_PATH, 'rb') as file:
    check_bundle = pickle.load(file)
assert len(check_bundle['annotations']) == 6766
assert {item['label'] for item in check_bundle['annotations']} == {0, 1}
example = check_bundle['annotations'][0]
print({key: value.shape if hasattr(value, 'shape') else value for key, value in example.items()})
print({name: len(ids) for name, ids in check_bundle['split'].items()})

## 4. Install MMAction2 1.2 environment

If the import cell requests a restart, restart the Kaggle session and rerun from the imports/configuration cells. Internet must be enabled for installation and checkpoint download.

In [ ]:
%pip uninstall -y mmaction2
%pip install -q 'mmengine>=0.10.0,<1.0' 'mmcv-lite>=2.0.0,<2.2.0' 'transformers==4.35.2'

In [ ]:
import mmcv
import mmengine
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('mmengine:', mmengine.__version__, '| mmcv:', mmcv.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before training.')
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
print('GPU:', gpu_name, '| compute capability:', gpu_capability)
if gpu_capability[0] < 7:
    raise RuntimeError(
        f'{gpu_name} has compute capability {gpu_capability}, but this Kaggle PyTorch build requires sm_70 or newer. '
        'In Kaggle Settings select a T4 GPU (T4 x2 is fine), restart the session, and rerun the notebook.'
    )

## 5. Obtain official PoseC3D config and create binary fine-tuning config

In [ ]:
MMACTION_REPO = Path('/kaggle/working/mmaction2')
if not MMACTION_REPO.exists():
    !git clone -q --depth 1 --branch v1.2.0 https://github.com/open-mmlab/mmaction2.git /kaggle/working/mmaction2
# MMAction2 v1.2.0 predates NumPy 2.0, where np.Inf was removed.
patched_files = 0
for source_path in MMACTION_REPO.rglob('*.py'):
    source_text = source_path.read_text(encoding='utf-8')
    compatible_text = source_text.replace('np.Inf', 'np.inf')
    if compatible_text != source_text:
        source_path.write_text(compatible_text, encoding='utf-8')
        patched_files += 1
print('NumPy 2 compatibility files patched:', patched_files)
%pip install -q -e /kaggle/working/mmaction2
import sys
if str(MMACTION_REPO) not in sys.path:
    sys.path.insert(0, str(MMACTION_REPO))
import mmaction
import transformers
print('mmaction:', mmaction.__version__, '| source:', mmaction.__file__)
print('transformers:', transformers.__version__)
assert str(MMACTION_REPO) in str(Path(mmaction.__file__)), mmaction.__file__
assert transformers.__version__ == '4.35.2', transformers.__version__
BASE_CONFIG = MMACTION_REPO / 'configs/skeleton/posec3d/slowonly_r50_8xb16-u48-240e_ntu60-xsub-keypoint.py'
assert BASE_CONFIG.exists(), BASE_CONFIG
print(BASE_CONFIG)

In [ ]:
from mmengine.config import Config

cfg = Config.fromfile(BASE_CONFIG)
cfg.model.cls_head.num_classes = 2
cfg.load_from = ('https://download.openmmlab.com/mmaction/v1.0/skeleton/posec3d/'
                 'slowonly_r50_8xb16-u48-240e_ntu60-xsub-keypoint/'
                 'slowonly_r50_8xb16-u48-240e_ntu60-xsub-keypoint_20220815-38db104b.pth')
cfg.work_dir = str(ARTIFACT_ROOT / 'work_dir')
cfg.randomness = dict(seed=SEED, deterministic=False)

def pose_dataset(split, pipeline, test_mode=False):
    return dict(type='PoseDataset', ann_file=str(ANNOTATION_PATH), split=split, pipeline=pipeline, test_mode=test_mode)

cfg.train_dataloader = dict(batch_size=8, num_workers=2, persistent_workers=True, sampler=dict(type='DefaultSampler', shuffle=True), dataset=pose_dataset('train', cfg.train_pipeline))
cfg.val_dataloader = dict(batch_size=8, num_workers=2, persistent_workers=True, sampler=dict(type='DefaultSampler', shuffle=False), dataset=pose_dataset('val', cfg.val_pipeline, True))
cfg.test_dataloader = dict(batch_size=4, num_workers=2, persistent_workers=True, sampler=dict(type='DefaultSampler', shuffle=False), dataset=pose_dataset('test', cfg.val_pipeline, True))
cfg.train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=20, val_begin=1, val_interval=1)
cfg.optim_wrapper.optimizer.lr = 0.01
cfg.param_scheduler[0].T_max = 20
cfg.default_hooks.checkpoint = dict(type='CheckpointHook', interval=1, save_best='auto', max_keep_ckpts=3)
cfg.val_evaluator = [dict(type='AccMetric')]
cfg.test_evaluator = [dict(type='AccMetric')]
cfg.log_processor = dict(type='LogProcessor', window_size=20, by_epoch=True)

CONFIG_PATH = ARTIFACT_ROOT / 'posec3d_fall_binary.py'
cfg.dump(CONFIG_PATH)
print('Saved config:', CONFIG_PATH)
print('Work dir:', cfg.work_dir)

## 6. Fine-tune PoseC3D

Run this cell once. The official training entry point validates every epoch and preserves the best checkpoint.

In [ ]:
!env PYTHONPATH=/kaggle/working/mmaction2 python /kaggle/working/mmaction2/tools/train.py {CONFIG_PATH}

## 7. Select checkpoint and dump held-out test predictions

In [ ]:
WORK_DIR = Path(cfg.work_dir)
best_candidates = sorted(WORK_DIR.glob('best*.pth'), key=lambda p: p.stat().st_mtime)
if best_candidates:
    BEST_CHECKPOINT = best_candidates[-1]
else:
    epoch_candidates = sorted(WORK_DIR.glob('epoch_*.pth'), key=lambda p: int(re.search(r'epoch_(\d+)', p.name).group(1)))
    if not epoch_candidates:
        raise FileNotFoundError('No trained checkpoint found.')
    BEST_CHECKPOINT = epoch_candidates[-1]
SOURCE_BEST_CHECKPOINT = BEST_CHECKPOINT
# PyTorch 2.6+ defaults torch.load to weights_only=True. Strip MMEngine's
# trusted training-history objects and preserve only the tensor state dict.
trusted_checkpoint = torch.load(
    SOURCE_BEST_CHECKPOINT, map_location='cpu', weights_only=False
)
BEST_CHECKPOINT = ARTIFACT_ROOT / 'posec3d_fall_weights.pth'
torch.save({'state_dict': trusted_checkpoint['state_dict']}, BEST_CHECKPOINT)
print('Selected training checkpoint:', SOURCE_BEST_CHECKPOINT)
print('Safe weights-only checkpoint:', BEST_CHECKPOINT)
TEST_RESULTS_PATH = ARTIFACT_ROOT / 'test_predictions.pkl'

### Export the selected binary PoseC3D checkpoint to ONNX

The exported model accepts a float32 tensor shaped `N x 17 x 48 x 64 x 64` and returns `[P(No Fall), P(Fall)]`.

In [ ]:
%pip install -q 'onnx>=1.16,<2' 'onnxruntime>=1.18,<2'

In [ ]:
import onnx
import onnxruntime as ort
from mmaction.apis import init_recognizer

export_source = init_recognizer(CONFIG_PATH, BEST_CHECKPOINT, device='cpu')

class PoseC3DDeployment(torch.nn.Module):
    def __init__(self, recognizer):
        super().__init__()
        self.backbone = recognizer.backbone
        self.cls_head = recognizer.cls_head

    def forward(self, pose_heatmaps):
        features = self.backbone(pose_heatmaps)
        logits = self.cls_head(features)
        return torch.softmax(logits, dim=1)

deployment_model = PoseC3DDeployment(export_source).eval()
dummy_input = torch.randn(1, 17, 48, 64, 64, dtype=torch.float32)
ONNX_PATH = ARTIFACT_ROOT / 'posec3d_fall.onnx'
with torch.no_grad():
    pytorch_output = deployment_model(dummy_input).numpy()

torch.onnx.export(
    deployment_model,
    dummy_input,
    ONNX_PATH,
    input_names=['pose_heatmaps'],
    output_names=['probabilities'],
    dynamic_axes={'pose_heatmaps': {0: 'batch'}, 'probabilities': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
)
onnx.checker.check_model(onnx.load(ONNX_PATH))
ort_session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
onnx_output = ort_session.run(None, {'pose_heatmaps': dummy_input.numpy()})[0]
np.testing.assert_allclose(onnx_output, pytorch_output, rtol=1e-3, atol=1e-4)
print('ONNX verified:', ONNX_PATH)
print('Input:', ort_session.get_inputs()[0].shape)
print('Output:', ort_session.get_outputs()[0].shape)
print('Maximum PyTorch/ONNX difference:', float(np.max(np.abs(onnx_output - pytorch_output))))

In [ ]:
!env PYTHONPATH=/kaggle/working/mmaction2 python /kaggle/working/mmaction2/tools/test.py {CONFIG_PATH} {BEST_CHECKPOINT} --dump {TEST_RESULTS_PATH}

## 8. Binary fall metrics and operating threshold

In [ ]:
with open(TEST_RESULTS_PATH, 'rb') as file:
    dumped = pickle.load(file)

def to_numpy(value):
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    return np.asarray(value)

scores, labels = [], []
for item in dumped:
    if hasattr(item, 'pred_score'):
        score = item.pred_score
        label = item.gt_label
    elif isinstance(item, dict):
        score = item.get('pred_score', item.get('pred_scores'))
        label = item.get('gt_label', item.get('label'))
    else:
        raise TypeError(f'Unsupported result type: {type(item)}')
    scores.append(to_numpy(score).reshape(-1))
    labels.append(int(to_numpy(label).reshape(-1)[0]))

scores = np.stack(scores)
labels = np.asarray(labels)
fall_probability = scores[:, 1]
predictions = (fall_probability >= 0.5).astype(int)

metrics = {
    'threshold': 0.5,
    'fall_precision': precision_score(labels, predictions, zero_division=0),
    'fall_recall': recall_score(labels, predictions, zero_division=0),
    'fall_f1': f1_score(labels, predictions, zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(labels, predictions),
    'pr_auc_average_precision': average_precision_score(labels, fall_probability),
}
print(json.dumps(metrics, indent=2))
print(classification_report(labels, predictions, target_names=['No Fall', 'Fall'], digits=4))
ConfusionMatrixDisplay(confusion_matrix(labels, predictions), display_labels=['No Fall', 'Fall']).plot(cmap='Blues')
plt.show()

with open(ARTIFACT_ROOT / 'test_metrics.json', 'w') as file:
    json.dump(metrics, file, indent=2)

In [ ]:
precision, recall, thresholds = precision_recall_curve(labels, fall_probability)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_index = int(np.argmax(f1_values))
print({'best_test_f1_threshold_FOR_ANALYSIS_ONLY': float(thresholds[best_index]), 'f1': float(f1_values[best_index]), 'precision': float(precision[best_index]), 'recall': float(recall[best_index])})
plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel('Fall recall')
plt.ylabel('Fall precision')
plt.title('Held-out test precision–recall curve')
plt.grid(alpha=0.3)
plt.show()
print('Do not select the deployment threshold on test data; choose it on validation predictions later.')

## 9. Preserve artifacts

Before the Kaggle session ends, create a version with outputs and download the best checkpoint, config, split metadata, annotation pickle, and metrics from `/kaggle/working/posec3d_fall/`. The held-out test set must not be used for model or threshold selection.